# Go1 ERFI study (RunPod)

Blind flat-ground replication of Campanaro et al., *Learning and Deploying Robust Locomotion
Policies with Minimal Dynamics Randomization*, on the Unitree Go1. Six training conditions
(no randomization, dynamics randomization, RFI, RAO, ERFI-C, ERFI-50), three seeds each,
evaluated under the paper's perturbation protocol (payload, push, friction, gravity, Kp).

Everything real lives in the repo: [`envs/erfi.py`](../../src/rl_locomotion/envs/erfi.py),
[`training/ppo.py`](../../src/rl_locomotion/training/ppo.py),
[`eval/perturb.py`](../../src/rl_locomotion/eval/perturb.py),
[`configs/experiment/erfi_study.yaml`](../../configs/experiment/erfi_study.yaml).
This notebook only drives it and looks at the results.

**Setup**: pod with an RTX 4090 and a network volume at `/workspace`, then
`bash runpod/bootstrap.sh` (installs the pinned requirements and clones the repo).
Budget: ~8 min per run on a 4090, 18 runs, so about 2.5 hours for the full sweep.

In [ ]:
# @title 1. Environment check
import os, subprocess, sys
from pathlib import Path

REPO = Path("/workspace/RL-for-Locomotion") if Path("/workspace").is_dir() else Path.cwd().parents[1]
os.chdir(REPO)
sys.path.insert(0, str(REPO / "src"))
os.environ["MUJOCO_GL"] = "egl"
os.environ["XLA_FLAGS"] = os.environ.get("XLA_FLAGS", "") + " --xla_gpu_triton_gemm_any=True"

subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], check=True)
import jax
print("jax", jax.__version__, jax.devices())
assert jax.devices()[0].platform == "gpu", "JAX is on CPU: run runpod/bootstrap.sh and restart the kernel"
print("repo:", REPO)

## 2. Smoke test (~3 min)

Two conditions, 2M steps each, then the evaluation on two perturbations. Proves the whole
chain on this pod before committing hours to it. Output goes to `erfi_study_smoke/`.

In [ ]:
!python scripts/train.py --conditions none erfi_50 --seeds 0 --smoke
!python scripts/eval.py --runs /workspace/experiments/erfi_study_smoke --n-episodes 10 --params payload_kg push_N --plot

## 3. Full sweep

Run this **in tmux from a terminal**, not here, so a dropped browser tab cannot kill it:

```bash
tmux new -s erfi
cd /workspace/RL-for-Locomotion
python scripts/train.py --config configs/experiment/erfi_study.yaml 2>&1 | tee /workspace/experiments/erfi_train.log
# Ctrl-B D to detach; tmux attach -t erfi to come back
```

Finished runs are skipped, so the same command resumes after a pod restart. The cell below
shows progress while it runs.

In [ ]:
# @title Progress of the sweep
import json
import pandas as pd

ROOT = Path("/workspace/experiments/erfi_study")
rows = []
for run in sorted(ROOT.glob("*/seed*")):
    curve = json.loads((run / "curve.json").read_text()) if (run / "curve.json").exists() else []
    rows.append({
        "run": str(run.relative_to(ROOT)),
        "finished": (run / "params_final").exists(),
        "evals": len(curve),
        "last_step": curve[-1]["step"] if curve else 0,
        "reward": round(curve[-1]["reward"], 3) if curve else None,
        "minutes": round(curve[-1]["wall_s"] / 60, 1) if curve else None,
    })
pd.DataFrame(rows) if rows else print("no runs yet")

In [ ]:
# @title Training curves, one line per condition (mean over seeds)
import matplotlib.pyplot as plt
from rl_locomotion.eval.perturb import CONDITION_COLORS, CONDITION_NAMES

fig, ax = plt.subplots(figsize=(8, 4))
for cond in CONDITION_COLORS:
    curves = [json.loads(p.read_text()) for p in ROOT.glob(f"{cond}/seed*/curve.json")]
    if not curves:
        continue
    n = min(len(c) for c in curves)
    steps = [c["step"] for c in curves[0][:n]]
    rewards = pd.DataFrame([[c[i]["reward"] for i in range(n)] for c in curves])
    ax.plot(steps, rewards.mean(), color=CONDITION_COLORS[cond], lw=2, label=CONDITION_NAMES[cond])
    ax.fill_between(steps, rewards.min(), rewards.max(), color=CONDITION_COLORS[cond], alpha=0.12, lw=0)
ax.set(xlabel="environment steps", ylabel="eval episode reward")
ax.grid(alpha=0.25)
ax.legend(frameon=False)
plt.show()

## 4. Evaluation

Runs the perturbation protocol on every finished run. 50 episodes per (run, parameter, level),
one JIT compile per run, so this takes a few minutes for the whole sweep. Already-evaluated
runs are skipped. Results: `results.csv`, `summary_success_rate.csv`, and the three curve figures.

In [ ]:
!python scripts/eval.py --config configs/experiment/erfi_study.yaml --plot

In [ ]:
# @title Results
from IPython.display import Image, display
from rl_locomotion.eval import perturb

results = pd.read_csv(ROOT / "results.csv")
display(perturb.summary_table(results, "success_rate"))
display(perturb.summary_table(results, "fall_rate"))
display(Image(filename=str(ROOT / "success_rate_curves.png")))

## 5. Watch a policy

Render one run under a chosen perturbation to see what the numbers mean.

In [ ]:
# @title Render one episode
import functools
import jax, jax.numpy as jp, numpy as np
import mediapy as media
import mujoco
from rl_locomotion.envs import erfi
from rl_locomotion.training import ppo

RUN = ROOT / "erfi_50" / "seed0"
PARAM, LEVEL = "payload_kg", 4.0   # any entry of perturb.PROTOCOL

env = erfi.load(perturb.eval_env_config(ppo.load_env_config(RUN), impl="jax"))
policy = ppo.load_policy(RUN, env)
env._mjx_model = perturb.perturbed_model(env, PARAM, LEVEL)

jit_reset, jit_step, jit_policy = jax.jit(env.reset), jax.jit(env.step), jax.jit(policy)
rng = jax.random.PRNGKey(0)
state = jit_reset(rng)
command = jp.array([0.5, 0.0, 0.0])
rollout = []
for _ in range(400):
    state.info["command"] = command
    rng, k = jax.random.split(rng)
    act, _ = jit_policy(state.obs, k)
    state = jit_step(state, act)
    rollout.append(state)
    if float(state.done):
        break
progress = float(state.data.xpos[env._torso_body_id][0] - rollout[0].data.xpos[env._torso_body_id][0])
print(f"{RUN.relative_to(ROOT)}  {PARAM}={LEVEL}: {len(rollout)} steps, fell={bool(state.done)}, progress {progress:.2f} m")

frames = env.render(rollout[::2], camera="track", width=640, height=480)
media.show_video(frames, fps=25, loop=False)